In [5]:
import pandas as pd

# Define the file paths
excel_file_path = r"C:\Users\rchan\Downloads\Table-5a-India-District-MPI-2018-1.xlsx"
district_file_path = "district_wise_mpi.csv" 

df_mpi_summary = pd.read_excel(excel_file_path, sheet_name='5a.1 India MPI District ', skiprows=4)
df_mpi_raw = pd.read_excel(excel_file_path, sheet_name='5a.4 District Raw Headcount', skiprows=6)

# Read the district CSV file (remains read_csv since it's still a CSV)
df_district = pd.read_csv(district_file_path)

print("MPI Summary Columns:")
print(df_mpi_summary.head(2))
print("\nMPI Raw Columns:")
print(df_mpi_raw.head(2))
print("\nDistrict Summary Columns:")
print(df_district.head(2))

MPI Summary Columns:
  State District Country MPI data source Unnamed: 4  Unnamed: 5  \
0   NaN      NaN     NaN             NaN        NaN         NaN   
1   NaN      NaN     NaN          Survey       Year         NaN   

   Population Share of the District  Unnamed: 7  \
0                               NaN         NaN   
1                               NaN         NaN   

   Multidimensional Poverty Index (MPI) of the country  Unnamed: 9  \
0                                                NaN           NaN   
1                                                NaN           NaN   

     Multidimensional poverty of the districts  \
0  Multidimensional Poverty Index\n(MPI = H*A)   
1                                          NaN   

                                         Unnamed: 11  \
0  Headcount ratio: Population in multidimensiona...   
1                                                NaN   

                                     Unnamed: 12             Unnamed: 13  \
0  Intensity of 

In [7]:
import pandas as pd
import numpy as np

# 1. Define the local file path
excel_file_path = r"C:\Users\rchan\Downloads\Table-5a-India-District-MPI-2018-1.xlsx"
# Update this path if the CSV is also in your Downloads
district_csv_path = "district_wise_mpi.csv" 

df_mpi_summary_raw = pd.read_excel(excel_file_path, sheet_name='5a.1 India MPI District ', header=None)

print("MPI Summary snippet:")
print(df_mpi_summary_raw.iloc[4:8, 0:13])

df_mpi_raw_raw = pd.read_excel(excel_file_path, sheet_name='5a.4 District Raw Headcount', header=None)

print("\nMPI Raw snippet:")
# iloc index remains the same as the structure is preserved from CSV
print(df_mpi_raw_raw.iloc[6:10, [0, 1, 16]])

# 4. Read the standalone CSV
df_district = pd.read_csv(district_csv_path)

print("\nDistrict Summary Head:")
print(df_district.head(2))

MPI Summary snippet:
      0         1        2                3     4   5   \
4  State  District  Country  MPI data source   NaN NaN   
5    NaN       NaN      NaN              NaN   NaN NaN   
6    NaN       NaN      NaN           Survey  Year NaN   
7    NaN       NaN      NaN              NaN   NaN NaN   

                                 6   7   \
4  Population Share of the District NaN   
5                               NaN NaN   
6                               NaN NaN   
7                               NaN NaN   

                                                  8   9   \
4  Multidimensional Poverty Index (MPI) of the co... NaN   
5                                                NaN NaN   
6                                                NaN NaN   
7                                                NaN NaN   

                                            10  \
4    Multidimensional poverty of the districts   
5  Multidimensional Poverty Index\n(MPI = H*A)   
6                    

In [9]:
import pandas as pd
import numpy as np

# 1. Define the local file paths
excel_path = r"C:\Users\rchan\Downloads\Table-5a-India-District-MPI-2018-1.xlsx"
# Assuming district_wise_mpi.csv is in your working directory or same folder
district_csv_path = "district_wise_mpi.csv" 

# 2. Read MPI Data from Excel sheets
df_mpi_summary = pd.read_excel(excel_path, sheet_name='5a.1 India MPI District ', skiprows=8, header=None)
df_mpi_summary = df_mpi_summary[[0, 1, 10, 11]].rename(columns={0: 'State', 1: 'District', 10: 'MPI', 11: 'Headcount_Ratio (H)'})

df_mpi_raw = pd.read_excel(excel_path, sheet_name='5a.4 District Raw Headcount', skiprows=8, header=None)
df_mpi_raw = df_mpi_raw[[0, 1, 16]].rename(columns={0: 'State', 1: 'District', 16: 'Electricity_Deprivation'})

# 3. Read the District Census/Summary CSV
df_dist = pd.read_csv(district_csv_path)

# 4. Standardize names for merging
for df in [df_mpi_summary, df_mpi_raw, df_dist]:
    df['State'] = df['State'].astype(str).str.strip().str.title()
    df['District'] = df['District'].astype(str).str.strip().str.title()

# 5. Integrate datasets
df_mpi = pd.merge(df_mpi_summary, df_mpi_raw, on=['State', 'District'], how='inner')
df_merged = pd.merge(df_mpi, df_dist, on=['State', 'District'], how='inner')

# 6. Deriving the final features (Your Logic)
df_merged['Electricity Access'] = 100 - df_merged['Electricity_Deprivation']
df_merged['Average Household Size'] = df_merged['Total Population'] / df_merged['Households']

# Create a proxy for Average Annual Income
# Assuming base average annual wage to create a realistic-looking spread: ₹ 1,50,000 per worker
df_merged['Average Annual Income (Proxy)'] = (df_merged['Total Working Population'] / df_merged['Total Population']) * 150000

# Handle Outliers in Income using IQR
Q1 = df_merged['Average Annual Income (Proxy)'].quantile(0.25)
Q3 = df_merged['Average Annual Income (Proxy)'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Capping outliers to reduce bias
df_merged['Average Annual Income'] = np.clip(df_merged['Average Annual Income (Proxy)'], lower_bound, upper_bound)

# Selecting necessary columns
final_cols = [
    'State', 'District', 
    'Total Population', 'Households', 'Total Working Population', 'Electricity_Deprivation',
    'Average Household Size', 'Electricity Access', 'Average Annual Income', 
    'MPI', 'Headcount_Ratio (H)'
]

df_final = df_merged[final_cols]

# 7. Save to CSV
df_final.to_csv('integrated_poverty_dataset.csv', index=False)

print("Saved integrated_poverty_dataset.csv with shape:", df_final.shape)
print(df_final.head())

Saved integrated_poverty_dataset.csv with shape: (576, 11)
                         State       District  Total Population  Households  \
0  Andaman And Nicobar Islands       Nicobars             36842        9288   
1  Andaman And Nicobar Islands  South Andaman            238142       59064   
2               Andhra Pradesh      Anantapur           4081148      968160   
3               Andhra Pradesh       Chittoor           4174064     1039953   
4               Andhra Pradesh  East Godavari           5154296     1428528   

   Total Working Population  Electricity_Deprivation  Average Household Size  \
0                     17125                  0.44087                3.966624   
1                     96831                  1.21042                4.031931   
2                   2036166                  0.53208                4.215365   
3                   1933357                  0.64508                4.013704   
4                   2093681                  0.60047              